In [1]:

import torch

print("Version de PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
    print(
        "Mémoire GPU totale :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "Go"
    )
else:
    print("Aucun GPU détecté : vérifie les paramètres d'exécution de Colab.")


Version de PyTorch : 2.11.0+cu130
GPU disponible : True
Nom du GPU : Tesla T4
Mémoire GPU totale : 14.56 Go


In [2]:

%pip install -q datasets transformers accelerate pandas scikit-learn matplotlib


Data

In [3]:

from datasets import load_dataset

dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

print(dataset)
print("\nColonnes disponibles :", dataset["train"].column_names)
print("Nombre d'exemples :", len(dataset["train"]))


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

Colonnes disponibles : ['pubid', 'question', 'context', 'long_answer', 'final_decision']
Nombre d'exemples : 1000


Explorer un exemple

In [4]:

example = dataset["train"][0]

print("Clés de l'exemple :", example.keys())
print("\nQuestion :")
print(example["question"])

print("\nContexte scientifique :")
print(example["context"])

print("\nRéponse attendue :")
print(example["final_decision"])


Clés de l'exemple : dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])

Question :
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Contexte scientifique :
{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into thre

In [5]:

import pandas as pd

df = dataset["train"].to_pandas()

print("Dimensions du jeu de données :", df.shape)
print("\nColonnes :")
print(df.columns.tolist())

print("\nExemples par classe :")
print(df["final_decision"].value_counts())

print("\nValeurs manquantes :")
print(df[["question", "context", "final_decision"]].isna().sum())


Dimensions du jeu de données : (1000, 5)

Colonnes :
['pubid', 'question', 'context', 'long_answer', 'final_decision']

Exemples par classe :
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64

Valeurs manquantes :
question          0
context           0
final_decision    0
dtype: int64


In [6]:

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

SEED = 42

# Vérifier les données nécessaires
required_columns = ["question", "context", "final_decision"]
assert all(col in df.columns for col in required_columns)
assert df[required_columns].notna().all().all()

# Indices de tous les exemples
indices = list(range(len(df)))
labels = df["final_decision"].tolist()

# 70 % pour l'entraînement, 30 % temporairement réservés
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

# Répartir les 30 % restants en 10 % validation et 20 % test
temp_labels = [labels[i] for i in temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=2/3,
    random_state=SEED,
    stratify=temp_labels
)

# Construire les trois sous-ensembles
df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

splits = DatasetDict({
    "train": Dataset.from_pandas(df_train, preserve_index=False),
    "validation": Dataset.from_pandas(df_val, preserve_index=False),
    "test": Dataset.from_pandas(df_test, preserve_index=False)
})

print(splits)


DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 700
    })
    validation: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 100
    })
    test: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 200
    })
})


In [7]:

train_set = set(train_idx)
val_set = set(val_idx)
test_set = set(test_idx)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)

print("Aucun chevauchement entre les partitions.")
print("\nRépartition des classes :")

for name, part in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    print(f"\n{name} ({len(part)} exemples)")
    print(
        part["final_decision"]
        .value_counts(normalize=True)
        .round(3)
    )


Aucun chevauchement entre les partitions.

Répartition des classes :

Train (700 exemples)
final_decision
yes      0.551
no       0.339
maybe    0.110
Name: proportion, dtype: float64

Validation (100 exemples)
final_decision
yes      0.55
no       0.34
maybe    0.11
Name: proportion, dtype: float64

Test (200 exemples)
final_decision
yes      0.555
no       0.335
maybe    0.110
Name: proportion, dtype: float64


In [8]:

import json
from pathlib import Path

output_dir = Path("/content/slm_medical")
output_dir.mkdir(parents=True, exist_ok=True)

splits.save_to_disk(str(output_dir / "pubmedqa_splits"))

config = {
    "dataset": "qiaojin/PubMedQA",
    "configuration": "pqa_labeled",
    "seed": SEED,
    "train_fraction": 0.70,
    "validation_fraction": 0.10,
    "test_fraction": 0.20,
    "target_column": "final_decision",
    "primary_metrics": ["accuracy", "macro_f1"]
}

with open(output_dir / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Partitions et configuration sauvegardées dans :", output_dir)


Saving the dataset (0/1 shards):   0%|          | 0/700 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Partitions et configuration sauvegardées dans : /content/slm_medical


# Charger le modèle en précision FP16 Qween

In [9]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Charger le tokenizer : il transforme le texte en tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Charger les poids du modèle en précision FP16 sur le GPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

# Passer en mode évaluation
model.eval()

print("Modèle chargé :", MODEL_NAME)
print("Appareil :", next(model.parameters()).device)
print(
    "Mémoire GPU allouée :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "Go"
)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modèle chargé : Qwen/Qwen2.5-1.5B-Instruct
Appareil : cuda:0
Mémoire GPU allouée : 2.88 Go


TEST

In [10]:

example = splits["test"][0]

prompt = f"""Answer the biomedical question using the scientific context.
Choose exactly one answer: yes, no, or maybe.

Context:
{example["context"]}

Question:
{example["question"]}

Answer:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Ne conserver que les tokens générés, sans répéter le prompt
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("Question :", example["question"])
print("Réponse attendue :", example["final_decision"])
print("Réponse du modèle :", repr(prediction))


Question : Antiretroviral therapy related adverse effects: Can sub-Saharan Africa cope with the new "test and treat" policy of the World Health Organization?
Réponse attendue : no
Réponse du modèle : ' Yes\nExplanation: Sub'


#Fiabiliser le format de sortie

In [11]:

import re
import torch

def predict_label(question, context):
    prompt = f"""You are classifying biomedical research questions.
Use the context to answer the question.
Return exactly one label: yes, no, or maybe.
Do not provide an explanation.

Context:
{context}

Question:
{question}

Label:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    text = text.strip().lower()

    # Accepter uniquement une classe au début de la sortie
    match = re.match(r"^(yes|no|maybe)\b", text)

    if match:
        return match.group(1), text

    return "invalid", text


example = splits["test"][0]

predicted_label, raw_output = predict_label(
    example["question"],
    example["context"]
)

print("Réponse attendue :", example["final_decision"])
print("Classe prédite :", predicted_label)
print("Sortie brute :", repr(raw_output))


Réponse attendue : no
Classe prédite : yes
Sortie brute : 'yes\n\nthis question'


#Évaluer le modèle de référence

In [12]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_true = []
y_pred = []
raw_outputs = []
latencies = []

# Évaluation sur le jeu de validation
for example in splits["validation"]:
    start = time.perf_counter()

    label, raw = predict_label(
        example["question"],
        example["context"]
    )

    latencies.append(time.perf_counter() - start)
    y_true.append(example["final_decision"].lower())
    y_pred.append(label)
    raw_outputs.append(raw)

# Calcul des métriques
accuracy = accuracy_score(y_true, y_pred)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=["yes", "no", "maybe"],
    average="macro",
    zero_division=0
)

print(f"Accuracy : {accuracy:.3f}")
print(f"Macro-F1 : {macro_f1:.3f}")
print(f"Latence médiane : {pd.Series(latencies).median():.3f} s")
print(
    "Taux de sorties invalides : "
    f"{sum(p not in ['yes', 'no', 'maybe'] for p in y_pred) / len(y_pred):.1%}"
)

print("\nRapport de classification :")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["yes", "no", "maybe"],
        zero_division=0
    )
)

Accuracy : 0.520
Macro-F1 : 0.259
Latence médiane : 0.354 s
Taux de sorties invalides : 0.0%

Rapport de classification :
              precision    recall  f1-score   support

         yes       0.54      0.91      0.68        55
          no       0.25      0.06      0.10        34
       maybe       0.00      0.00      0.00        11

    accuracy                           0.52       100
   macro avg       0.26      0.32      0.26       100
weighted avg       0.38      0.52      0.41       100



#quantification 4 bits

In [13]:
baseline_val = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "raw_output": raw_outputs,
    "latency_seconds": latencies
})

baseline_val.to_csv(
    "/content/slm_medical/baseline_validation.csv",
    index=False
)

print("Résultats de référence sauvegardés.")

Résultats de référence sauvegardés.


In [14]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.8 MB/s eta 0:00:00


In [15]:
import gc
import torch

# Supprimer le modèle FP16 de la mémoire
del model

# Nettoyer les objets Python inutilisés
gc.collect()

# Libérer la mémoire GPU qui n'est plus utilisée
torch.cuda.empty_cache()

# Vérifier la mémoire GPU disponible
print(
    "Mémoire GPU utilisée :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "Go"
)
print(
    "Mémoire GPU réservée :",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "Go"
)

Mémoire GPU utilisée : 0.01 Go
Mémoire GPU réservée : 0.02 Go


In [16]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Définir la quantification 4 bits
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Charger le modèle quantifié
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

model_4bit.eval()

print("Modèle chargé en quantification 4 bits.")
print("GPU :", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modèle chargé en quantification 4 bits.
GPU : Tesla T4


In [17]:
print("Type du modèle :", type(model_4bit).__name__)
print(
    "Mémoire GPU allouée :",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "Go"
)

Type du modèle : Qwen2ForCausalLM
Mémoire GPU allouée : 1.08 Go


In [19]:
print("accuracy_4bit existe :", "accuracy_4bit" in globals())
print("macro_f1_4bit existe :", "macro_f1_4bit" in globals())
print("Prédictions disponibles :", "y_pred_4bit" in globals())
print("Latences disponibles :", "latencies_4bit" in globals())

accuracy_4bit existe : False
macro_f1_4bit existe : False
Prédictions disponibles : False
Latences disponibles : False


In [20]:
print("Modèle 4 bits chargé :", "model_4bit" in globals())
print("Tokenizer disponible :", "tokenizer" in globals())
print("Jeu de validation disponible :", "splits" in globals())

Modèle 4 bits chargé : True
Tokenizer disponible : True
Jeu de validation disponible : True


In [21]:
import time
import re
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from IPython.display import display

y_true_4bit = []
y_pred_4bit = []
raw_outputs_4bit = []
latencies_4bit = []

def predict_label_4bit(question, context):
    prompt = f"""You are classifying biomedical research questions.
Use the context to answer the question.
Return exactly one label: yes, no, or maybe.
Do not provide an explanation.

Context:
{context}

Question:
{question}

Label:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.inference_mode():
        outputs = model_4bit.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip().lower()

    match = re.match(r"^(yes|no|maybe)\b", raw)
    return (match.group(1) if match else "invalid"), raw


total = len(splits["validation"])

for i, example in enumerate(splits["validation"]):
    start = time.perf_counter()

    label, raw = predict_label_4bit(
        example["question"],
        example["context"]
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latencies_4bit.append(time.perf_counter() - start)
    y_true_4bit.append(example["final_decision"].lower())
    y_pred_4bit.append(label)
    raw_outputs_4bit.append(raw)

    if (i + 1) % 10 == 0 or i + 1 == total:
        print(f"Exemples évalués : {i + 1}/{total}")

accuracy_4bit = accuracy_score(y_true_4bit, y_pred_4bit)

macro_f1_4bit = f1_score(
    y_true_4bit,
    y_pred_4bit,
    labels=["yes", "no", "maybe"],
    average="macro",
    zero_division=0
)

print(f"\nAccuracy 4 bits : {accuracy_4bit:.3f}")
print(f"Macro-F1 4 bits : {macro_f1_4bit:.3f}")
print(
    "Latence médiane :",
    round(pd.Series(latencies_4bit).median(), 3),
    "s"
)
print(
    "Taux de sorties invalides :",
    f"{sum(p == 'invalid' for p in y_pred_4bit) / total:.1%}"
)

print("\nRapport de classification :")
print(classification_report(
    y_true_4bit,
    y_pred_4bit,
    labels=["yes", "no", "maybe"],
    zero_division=0
))

Exemples évalués : 10/100
Exemples évalués : 20/100
Exemples évalués : 30/100
Exemples évalués : 40/100
Exemples évalués : 50/100
Exemples évalués : 60/100
Exemples évalués : 70/100
Exemples évalués : 80/100
Exemples évalués : 90/100
Exemples évalués : 100/100

Accuracy 4 bits : 0.530
Macro-F1 4 bits : 0.248
Latence médiane : 0.277 s
Taux de sorties invalides : 0.0%

Rapport de classification :
              precision    recall  f1-score   support

         yes       0.55      0.95      0.69        55
          no       0.20      0.03      0.05        34
       maybe       0.00      0.00      0.00        11

    accuracy                           0.53       100
   macro avg       0.25      0.32      0.25       100
weighted avg       0.37      0.53      0.40       100



La latence médiane diminue d'environ 22 %. L'accuracy gagne un point de pourcentage, mais le Macro-F1 baisse légèrement. Avec seulement 100 exemples de validation, nous ne pouvons pas conclure que cette différence d'accuracy reflète une véritable amélioration.

In [22]:
results_4bit = pd.DataFrame({
    "true_label": y_true_4bit,
    "predicted_label": y_pred_4bit,
    "raw_output": raw_outputs_4bit,
    "latency_seconds": latencies_4bit
})

results_4bit.to_csv(
    "/content/slm_medical/4bit_validation.csv",
    index=False
)

print("Résultats 4 bits sauvegardés.")

Résultats 4 bits sauvegardés.


#fine-tuning QLoRA

In [23]:

%pip install -q peft accelerate bitsandbytes


In [24]:

import torch
import transformers
import peft
import bitsandbytes

print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT :", peft.__version__)
print("BitsAndBytes :", bitsandbytes.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))


PyTorch : 2.11.0+cu130
Transformers : 5.20.1
PEFT : 0.21.2
BitsAndBytes : 0.50.2
GPU disponible : True
GPU : Tesla T4


#Préparer les données médicales

In [25]:

from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = 1024

# Vérifier que les données sont disponibles
print("Exemples d'entraînement :", len(splits["train"]))
print("Exemples de validation :", len(splits["validation"]))

# Le tokenizer doit disposer d'un token de padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prepare_example(example):
    # Le modèle reçoit le contexte et la question
    prompt = f"""Classify this biomedical question using the scientific context.
Return exactly one label: yes, no, or maybe.

Context:
{example["context"]}

Question:
{example["question"]}

Label:"""

    # Encoder séparément le prompt et la réponse attendue
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    answer_ids = tokenizer(
        " " + example["final_decision"].lower(),
        add_special_tokens=False
    )["input_ids"]

    # Ajouter le token de fin de séquence
    answer_ids = answer_ids + [tokenizer.eos_token_id]

    # Réserver de la place pour la réponse
    available_prompt_length = MAX_LENGTH - len(answer_ids)
    prompt_ids = prompt_ids[-available_prompt_length:]

    input_ids = prompt_ids + answer_ids

    # Calculer la perte uniquement sur la réponse,
    # pas sur le prompt fourni au modèle
    labels = [-100] * len(prompt_ids) + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

train_data = Dataset.from_list([
    prepare_example(example)
    for example in splits["train"]
])

val_data = Dataset.from_list([
    prepare_example(example)
    for example in splits["validation"]
])

print("\nDonnées préparées.")
print("Taille du train :", len(train_data))
print("Taille de la validation :", len(val_data))
print("Longueur du premier exemple :", len(train_data[0]["input_ids"]))
print("Réponse cible :", tokenizer.decode(
    [token for token in train_data[0]["labels"] if token != -100]
))


Exemples d'entraînement : 700
Exemples de validation : 100

Données préparées.
Taille du train : 700
Taille de la validation : 100
Longueur du premier exemple : 585
Réponse cible :  yes<|im_end|>


#Charger le modèle en 4 bits pour QLoRA

In [26]:

import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

# Configuration de la quantification
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Charger le modèle de base quantifié
model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Préparer le modèle pour l'entraînement quantifié
model_qlora.config.use_cache = False
model_qlora = prepare_model_for_kbit_training(model_qlora)

print("Modèle quantifié chargé et préparé.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modèle quantifié chargé et préparé.


#Ajouter les adaptateurs LoRA

In [27]:

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_qlora = get_peft_model(model_qlora, lora_config)

# Afficher le nombre de paramètres entraînables
model_qlora.print_trainable_parameters()


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [28]:

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
import inspect

# Vérifier la compatibilité des arguments avec Transformers installé
print("TrainingArguments accepte eval_strategy :",
      "eval_strategy" in inspect.signature(TrainingArguments).parameters)
print("TrainingArguments accepte evaluation_strategy :",
      "evaluation_strategy" in inspect.signature(TrainingArguments).parameters)


TrainingArguments accepte eval_strategy : True
TrainingArguments accepte evaluation_strategy : False


In [30]:

import inspect
from transformers import TrainingArguments

params = inspect.signature(TrainingArguments).parameters

for name in [
    "warmup_ratio",
    "warmup_steps",
    "eval_strategy",
    "save_strategy",
    "gradient_accumulation_steps",
    "fp16",
    "load_best_model_at_end"
]:
    print(f"{name}: {name in params}")


warmup_ratio: False
warmup_steps: True
eval_strategy: True
save_strategy: True
gradient_accumulation_steps: True
fp16: True
load_best_model_at_end: True


In [32]:
warmup_steps=10,

In [33]:

import gc
import inspect
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Libérer la mémoire de l'ancien modèle d'évaluation
if "model_4bit" in globals():
    del model_4bit

gc.collect()
torch.cuda.empty_cache()

# Préparer le remplissage des séquences
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_qlora,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

# Paramètres d'entraînement
training_args = TrainingArguments(
    output_dir="/content/slm_medical/qlora_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,  # Remplace warmup_ratio
    lr_scheduler_type="cosine",
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    save_total_limit=2,
    report_to="none",
    seed=42
)

# Construire le Trainer selon l'API disponible
trainer_kwargs = {
    "model": model_qlora,
    "args": training_args,
    "train_dataset": train_data,
    "eval_dataset": val_data,
    "data_collator": data_collator
}

trainer_parameters = inspect.signature(Trainer).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

# Lancer l'entraînement
train_result = trainer.train()

print("\nEntraînement terminé.")
print("Perte d'entraînement :", train_result.training_loss)

# Évaluer sur la validation
validation_metrics = trainer.evaluate()
print("\nMétriques de validation :")
print(validation_metrics)

# Sauvegarder le meilleur adaptateur
adapter_dir = "/content/slm_medical/qlora_adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("\nAdaptateur sauvegardé :", adapter_dir)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.217590,0.098563
2,0.152569,0.136347


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Entraînement terminé.
Perte d'entraînement : 0.8857763341882012


Training Loss,Validation Loss,Epoch
0.152569,0.098563,2



Métriques de validation :
{'eval_loss': 0.09856265783309937}

Adaptateur sauvegardé : /content/slm_medical/qlora_adapter


In [34]:

import time
import re
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Utiliser le modèle retenu par le Trainer
model_qlora = trainer.model
model_qlora.eval()

# Activer le cache pour accélérer la génération pendant l'inférence
model_qlora.config.use_cache = True

y_true_qlora = []
y_pred_qlora = []
raw_outputs_qlora = []
latencies_qlora = []

def predict_label_qlora(question, context):
    # Garder la même formulation que pendant l'entraînement
    prompt = f"""Classify this biomedical question using the scientific context.
Return exactly one label: yes, no, or maybe.

Context:
{context}

Question:
{question}

Label:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to("cuda")

    # Synchroniser le GPU pour mesurer correctement le temps
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model_qlora.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    latency = time.perf_counter() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip().lower()

    match = re.match(r"^(yes|no|maybe)\b", raw)
    label = match.group(1) if match else "invalid"

    return label, raw, latency


# Évaluation sur la validation
for i, example in enumerate(splits["validation"]):
    label, raw, latency = predict_label_qlora(
        example["question"],
        example["context"]
    )

    y_true_qlora.append(example["final_decision"].lower())
    y_pred_qlora.append(label)
    raw_outputs_qlora.append(raw)
    latencies_qlora.append(latency)

    if (i + 1) % 10 == 0:
        print(f"Exemples évalués : {i + 1}/100")

# Calcul des métriques
accuracy_qlora = accuracy_score(y_true_qlora, y_pred_qlora)

macro_f1_qlora = f1_score(
    y_true_qlora,
    y_pred_qlora,
    labels=["yes", "no", "maybe"],
    average="macro",
    zero_division=0
)

print("\n--- Résultats QLoRA ---")
print(f"Accuracy : {accuracy_qlora:.3f}")
print(f"Macro-F1 : {macro_f1_qlora:.3f}")
print(f"Latence médiane : {pd.Series(latencies_qlora).median():.3f} s")
print(
    "Taux de sorties invalides : "
    f"{sum(p == 'invalid' for p in y_pred_qlora) / len(y_pred_qlora):.1%}"
)

print("\nRapport de classification :")
print(classification_report(
    y_true_qlora,
    y_pred_qlora,
    labels=["yes", "no", "maybe"],
    zero_division=0
))

print("Matrice de confusion (lignes = vraies classes) :")
print(confusion_matrix(
    y_true_qlora,
    y_pred_qlora,
    labels=["yes", "no", "maybe"]
))


Exemples évalués : 10/100
Exemples évalués : 20/100
Exemples évalués : 30/100
Exemples évalués : 40/100
Exemples évalués : 50/100
Exemples évalués : 60/100
Exemples évalués : 70/100
Exemples évalués : 80/100
Exemples évalués : 90/100
Exemples évalués : 100/100

--- Résultats QLoRA ---
Accuracy : 0.940
Macro-F1 : 0.880
Latence médiane : 0.634 s
Taux de sorties invalides : 0.0%

Rapport de classification :
              precision    recall  f1-score   support

         yes       0.90      1.00      0.95        55
          no       1.00      0.97      0.99        34
       maybe       1.00      0.55      0.71        11

    accuracy                           0.94       100
   macro avg       0.97      0.84      0.88       100
weighted avg       0.95      0.94      0.93       100

Matrice de confusion (lignes = vraies classes) :
[[55  0  0]
 [ 1 33  0]
 [ 5  0  6]]


In [35]:

results_qlora = pd.DataFrame({
    "true_label": y_true_qlora,
    "predicted_label": y_pred_qlora,
    "raw_output": raw_outputs_qlora,
    "latency_seconds": latencies_qlora
})

results_qlora.to_csv(
    "/content/slm_medical/qlora_validation.csv",
    index=False
)

print("Prédictions QLoRA sauvegardées.")


Prédictions QLoRA sauvegardées.


le modèle adapté obtient 94 % d'accuracy et un Macro-F1 de 0,88 sur les 100 exemples de validation. C'est une amélioration importante par rapport aux modèles non adaptés. Le fine-tuning QLoRA apporte ici un gain de 42 points de pourcentage d'accuracy par rapport au modèle FP16 initial. Sa latence est toutefois plus élevée, ce qui rappelle que qualité et rapidité sont deux dimensions différentes.

#Harmoniser la comparaison

In [36]:

COMMON_MAX_LENGTH = 1024

def common_prompt(question, context):
    return f"""Classify this biomedical question using the scientific context.
Return exactly one label: yes, no, or maybe.

Context:
{context}

Question:
{question}

Label:"""

print("Prompt commun défini.")
print("Longueur maximale :", COMMON_MAX_LENGTH)
print("Exemples de validation :", len(splits["validation"]))


Prompt commun défini.
Longueur maximale : 1024
Exemples de validation : 100


In [37]:

import time
import re
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_model(model, dataset, model_name):
    model.eval()

    y_true = []
    y_pred = []
    raw_outputs = []
    latencies = []

    for example in dataset:
        prompt = common_prompt(
            example["question"],
            example["context"]
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=COMMON_MAX_LENGTH
        ).to("cuda")

        # Synchroniser le GPU pour mesurer correctement la latence
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        latency = time.perf_counter() - start

        # Récupérer uniquement les nouveaux tokens générés
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip().lower()

        match = re.match(r"^(yes|no|maybe)\b", raw)
        prediction = match.group(1) if match else "invalid"

        y_true.append(example["final_decision"].lower())
        y_pred.append(prediction)
        raw_outputs.append(raw)
        latencies.append(latency)

    results = {
        "Modèle": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true,
            y_pred,
            labels=["yes", "no", "maybe"],
            average="macro",
            zero_division=0
        ),
        "Latence médiane (s)": float(
            pd.Series(latencies).median()
        ),
        "Taux de sorties invalides": sum(
            p == "invalid" for p in y_pred
        ) / len(y_pred)
    }

    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "raw_output": raw_outputs,
        "latency_seconds": latencies
    })

    print(f"Résultats : {model_name}")
    print(f"Accuracy : {results['Accuracy']:.3f}")
    print(f"Macro-F1 : {results['Macro-F1']:.3f}")
    print(
        "Latence médiane : "
        f"{results['Latence médiane (s)']:.3f} s"
    )
    print(
        "Taux de sorties invalides : "
        f"{results['Taux de sorties invalides']:.1%}"
    )

    print("\nRapport par classe :")
    print(classification_report(
        y_true,
        y_pred,
        labels=["yes", "no", "maybe"],
        zero_division=0
    ))

    predictions_df.to_csv(
        "/content/slm_medical/qlora_common_validation.csv",
        index=False
    )

    return results, predictions_df


qlora_common_results, qlora_common_predictions = evaluate_model(
    model_qlora,
    splits["validation"],
    "QLoRA"
)


Résultats : QLoRA
Accuracy : 0.940
Macro-F1 : 0.880
Latence médiane : 0.257 s
Taux de sorties invalides : 0.0%

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.90      1.00      0.95        55
          no       1.00      0.97      0.99        34
       maybe       1.00      0.55      0.71        11

    accuracy                           0.94       100
   macro avg       0.97      0.84      0.88       100
weighted avg       0.95      0.94      0.93       100



#Évaluer le modèle FP16
Nous allons maintenant mesurer le modèle de base sans quantification ni fine-tuning. Cela permettra de comparer le modèle initial au QLoRA.

In [38]:

import gc
import torch
from transformers import AutoModelForCausalLM

# Sauvegarder les prédictions QLoRA avant de libérer sa mémoire
qlora_common_predictions.to_csv(
    "/content/slm_medical/qlora_common_validation.csv",
    index=False
)

# Libérer la mémoire GPU occupée par QLoRA
if "trainer" in globals():
    del trainer

if "model_qlora" in globals():
    del model_qlora

gc.collect()
torch.cuda.empty_cache()

print(
    "Mémoire GPU libre :",
    round(
        (torch.cuda.get_device_properties(0).total_memory
         - torch.cuda.memory_allocated(0)) / 1024**3,
        2
    ),
    "Go environ"
)

# Charger le modèle de base en FP16
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

model_fp16.eval()

# Évaluer avec le même prompt et le même protocole
fp16_common_results, fp16_common_predictions = evaluate_model(
    model_fp16,
    splits["validation"],
    "FP16"
)

# Sauvegarder les prédictions sous leur propre nom
fp16_common_predictions.to_csv(
    "/content/slm_medical/fp16_common_validation.csv",
    index=False
)

# Libérer la mémoire GPU
del model_fp16
gc.collect()
torch.cuda.empty_cache()

print("Évaluation FP16 terminée.")


Mémoire GPU libre : 13.03 Go environ


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Résultats : FP16
Accuracy : 0.510
Macro-F1 : 0.244
Latence médiane : 0.238 s
Taux de sorties invalides : 0.0%

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.54      0.91      0.68        55
          no       0.20      0.03      0.05        34
       maybe       0.00      0.00      0.00        11

    accuracy                           0.51       100
   macro avg       0.25      0.31      0.24       100
weighted avg       0.37      0.51      0.39       100

Évaluation FP16 terminée.


#Évaluer le modèle quantifié en 4 bits
Ce modèle utilise une quantification NF4 : l’objectif est de réduire la mémoire nécessaire tout en conservant des performances acceptables.

In [39]:

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Libérer toute mémoire GPU encore inutilisée
gc.collect()
torch.cuda.empty_cache()

# Configuration de la quantification 4 bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Charger le modèle quantifié
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model_4bit.eval()

# Évaluer avec le prompt commun sur les 100 exemples
int4_common_results, int4_common_predictions = evaluate_model(
    model_4bit,
    splits["validation"],
    "4-bit NF4"
)

# Sauvegarder les prédictions
int4_common_predictions.to_csv(
    "/content/slm_medical/4bit_common_validation.csv",
    index=False
)

# Relever la mémoire GPU allouée pendant que le modèle est chargé
gpu_memory_gb = torch.cuda.memory_allocated() / 1024**3
print(f"Mémoire GPU allouée : {gpu_memory_gb:.2f} Go")

# Libérer le modèle
del model_4bit
gc.collect()
torch.cuda.empty_cache()

print("Évaluation 4 bits terminée.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Résultats : 4-bit NF4
Accuracy : 0.540
Macro-F1 : 0.271
Latence médiane : 0.295 s
Taux de sorties invalides : 2.0%

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.57      0.95      0.71        55
          no       0.50      0.06      0.11        34
       maybe       0.00      0.00      0.00        11

   micro avg       0.55      0.54      0.55       100
   macro avg       0.36      0.33      0.27       100
weighted avg       0.48      0.54      0.42       100

Mémoire GPU allouée : 2.61 Go
Évaluation 4 bits terminée.


#Comment interpréter ces résultats ?

- FP16 : le modèle de base est rapide, mais ses prédictions sont faibles sur cette tâche.
- 4-bit NF4 : l’accuracy augmente de 3 points par rapport au FP16, mais le Macro-F1 reste faible. La classe maybe n’est toujours pas reconnue correctement.
- QLoRA : c’est de loin la meilleure configuration en qualité de prédiction. Elle atteint 94 % d’accuracy et un Macro-F1 de 0,880, avec une latence médiane de 0,257 seconde.
Le résultat le plus intéressant est que la quantification seule ne suffit pas à améliorer significativement les capacités du modèle, alors que le fine-tuning QLoRA semble apporter un gain important. Cela reste à confirmer sur le jeu de test indépendant.

#Évaluer sur le jeu de test

In [40]:

# Vérifier que le jeu de test contient bien 200 exemples
print("Nombre d'exemples de test :", len(splits["test"]))

# Vérifier la répartition des classes
print("\nRépartition des classes :")
print(splits["test"].to_pandas()["final_decision"].value_counts())

# Vérifier que le prompt commun est toujours défini
print("\nPrompt commun disponible :", callable(common_prompt))
print("Longueur maximale :", COMMON_MAX_LENGTH)

# Définir le dossier de sauvegarde des résultats
import os
os.makedirs("/content/slm_medical/final_test", exist_ok=True)

print("\nPréparation terminée.")


Nombre d'exemples de test : 200

Répartition des classes :
final_decision
yes      111
no        67
maybe     22
Name: count, dtype: int64

Prompt commun disponible : True
Longueur maximale : 1024

Préparation terminée.


In [41]:

import os
import gc
import time
import re
import pandas as pd
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

TEST_DIR = "/content/slm_medical/final_test"
os.makedirs(TEST_DIR, exist_ok=True)

test_dataset = splits["test"]

def evaluate_final_model(model, model_name):
    model.eval()

    y_true = []
    y_pred = []
    raw_outputs = []
    latencies = []

    # Réinitialiser les statistiques mémoire du GPU
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for i, example in enumerate(test_dataset):
        prompt = common_prompt(
            example["question"],
            example["context"]
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=COMMON_MAX_LENGTH
        ).to("cuda")

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        latency = time.perf_counter() - start

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip().lower()

        match = re.match(r"^(yes|no|maybe)\b", raw)
        prediction = match.group(1) if match else "invalid"

        y_true.append(example["final_decision"].lower())
        y_pred.append(prediction)
        raw_outputs.append(raw)
        latencies.append(latency)

        if (i + 1) % 50 == 0:
            print(f"{model_name} : {i + 1}/200 exemples évalués")

    peak_memory_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=["yes", "no", "maybe"],
        average="macro",
        zero_division=0
    )

    invalid_rate = sum(
        prediction == "invalid" for prediction in y_pred
    ) / len(y_pred)

    results = {
        "Modèle": model_name,
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Latence médiane (s)": float(
            pd.Series(latencies).median()
        ),
        "Taux sorties invalides": invalid_rate,
        "Pic mémoire GPU (Go)": peak_memory_gb
    }

    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "raw_output": raw_outputs,
        "latency_seconds": latencies
    })

    predictions_df.to_csv(
        f"{TEST_DIR}/{model_name}_predictions.csv",
        index=False
    )

    print(f"\n=== Résultats {model_name} ===")
    print(f"Accuracy : {accuracy:.3f}")
    print(f"Macro-F1 : {macro_f1:.3f}")
    print(f"Latence médiane : {results['Latence médiane (s)']:.3f} s")
    print(f"Sorties invalides : {invalid_rate:.1%}")
    print(f"Pic mémoire GPU : {peak_memory_gb:.2f} Go")
    print("\nRapport par classe :")
    print(classification_report(
        y_true,
        y_pred,
        labels=["yes", "no", "maybe"],
        zero_division=0
    ))

    return results


def cleanup_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


# 1. FP16
print("\nChargement du modèle FP16...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

fp16_test_results = evaluate_final_model(
    model_fp16, "FP16"
)
cleanup_model(model_fp16)


# 2. Quantification 4 bits
print("\nChargement du modèle 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

int4_test_results = evaluate_final_model(
    model_4bit, "4-bit-NF4"
)
cleanup_model(model_4bit)


# 3. QLoRA : recharger le modèle de base quantifié,
# puis l'adaptateur entraîné précédemment.
print("\nChargement du modèle QLoRA...")
from peft import PeftModel

base_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model_qlora_test = PeftModel.from_pretrained(
    base_qlora,
    "/content/slm_medical/qlora_adapter"
)
model_qlora_test.eval()

qlora_test_results = evaluate_final_model(
    model_qlora_test, "QLoRA"
)
cleanup_model(model_qlora_test)


# Tableau récapitulatif final
final_results = pd.DataFrame([
    fp16_test_results,
    int4_test_results,
    qlora_test_results
])

final_results.to_csv(
    f"{TEST_DIR}/final_comparison.csv",
    index=False
)

print("\n=== COMPARAISON FINALE SUR LE TEST ===")
display(final_results.round(4))
print(f"\nRésultats enregistrés dans : {TEST_DIR}")



Chargement du modèle FP16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FP16 : 50/200 exemples évalués
FP16 : 100/200 exemples évalués
FP16 : 150/200 exemples évalués
FP16 : 200/200 exemples évalués

=== Résultats FP16 ===
Accuracy : 0.550
Macro-F1 : 0.274
Latence médiane : 0.227 s
Sorties invalides : 0.5%
Pic mémoire GPU : 4.58 Go

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.57      0.97      0.72       111
          no       0.20      0.01      0.03        67
       maybe       0.33      0.05      0.08        22

   micro avg       0.55      0.55      0.55       200
   macro avg       0.37      0.34      0.27       200
weighted avg       0.42      0.55      0.42       200


Chargement du modèle 4-bit NF4...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit-NF4 : 50/200 exemples évalués
4-bit-NF4 : 100/200 exemples évalués
4-bit-NF4 : 150/200 exemples évalués
4-bit-NF4 : 200/200 exemples évalués

=== Résultats 4-bit-NF4 ===
Accuracy : 0.535
Macro-F1 : 0.271
Latence médiane : 0.306 s
Sorties invalides : 4.0%
Pic mémoire GPU : 5.68 Go

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.57      0.95      0.71       111
          no       0.33      0.01      0.03        67
       maybe       0.17      0.05      0.07        22

   micro avg       0.56      0.54      0.55       200
   macro avg       0.36      0.34      0.27       200
weighted avg       0.45      0.54      0.41       200


Chargement du modèle QLoRA...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

QLoRA : 50/200 exemples évalués
QLoRA : 100/200 exemples évalués
QLoRA : 150/200 exemples évalués
QLoRA : 200/200 exemples évalués

=== Résultats QLoRA ===
Accuracy : 0.905
Macro-F1 : 0.832
Latence médiane : 0.275 s
Sorties invalides : 0.5%
Pic mémoire GPU : 6.76 Go

Rapport par classe :
              precision    recall  f1-score   support

         yes       0.91      0.96      0.93       111
          no       0.97      0.93      0.95        67
       maybe       0.71      0.55      0.62        22

   micro avg       0.91      0.91      0.91       200
   macro avg       0.86      0.81      0.83       200
weighted avg       0.91      0.91      0.90       200


=== COMPARAISON FINALE SUR LE TEST ===


,Modèle,Accuracy,Macro-F1,Latence médiane (s),Taux sorties invalides,Pic mémoire GPU (Go)
0,FP16,0.550,0.2743,0.2272,0.005,4.5850
1,4-bit-NF4,0.535,0.2714,0.3058,0.040,5.6804
2,QLoRA,0.905,0.8321,0.2749,0.005,6.7641



Résultats enregistrés dans : /content/slm_medical/final_test


In [42]:

import shutil
import os

# Dossier du projet
dossier = "/content/slm_medical"

# Vérifier qu'il existe
if os.path.exists(dossier):
    zip_path = shutil.make_archive(
        "/content/slm_medical",
        "zip",
        root_dir="/content",
        base_dir="slm_medical"
    )
    print("ZIP créé :", zip_path)
else:
    print("Le dossier slm_medical est introuvable.")


ZIP créé : /content/slm_medical.zip
